## 0. Install dependencies



In [ ]:
%pip install unsloth pandas datasets


## 1. Configure system, summary type, finalized CSVs, and Qwen3-8B




In [ ]:
from pathlib import Path
import json, re, os, hashlib, time, uuid, inspect, importlib.metadata
import pandas as pd

# ---------------- Edit this cell ----------------
SYSTEM_KEY = "apache"  # These uploaded files are bgl, apache, linux, zookeeper
APPROACH = "stratified"  # "stratified" or "random"
SUMMARY_TYPE = "hybrid"  # "extractive", "abstractive", or "hybrid"
DATA_DIR = Path("/content")  # Upload these three CSVs to Colab, or change DATA_DIR.
TRAIN_FILE = DATA_DIR / "..."
VALIDATION_FILE = DATA_DIR / "/..."
TEST_FILE = DATA_DIR / "/..."

# All validation rows are used for checkpoint selection and evaluation.
VALIDATION_PARTITION_FILE = None
CHECKPOINT_FLAG_COLUMN = None


INPUT_COLUMN = "Content_without_labels"
TARGET_COLUMN = "GT"

MODEL_NAME = "unsloth/Qwen3-8B-bnb-4bit"
LOAD_IN_4BIT = True
LORA_R = 16
LORA_ALPHA = 16
LEARNING_RATE = 1e-4
NUM_TRAIN_EPOCHS = 3
BALANCE_TRAINING = True
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
EVAL_BATCH_SIZE = 1
SEED = 3407
# MAX_NEW_TOKENS = 5000

SUMMARY_TOKEN_CONFIG = {
    "extractive": (4096, 600),
    "abstractive": (4096, 600),
    "hybrid": (6144, 1500),
}
MAX_SEQ_LENGTH, MAX_NEW_TOKENS = SUMMARY_TOKEN_CONFIG[SUMMARY_TYPE]


OUTPUT_ROOT = Path("/content/logsumm_runs")
WINDOW_SIZE = None

def resolve_window_size(paths, explicit=None):
    filename_sizes = set()
    data_sizes = set()
    for path in paths:
        path = Path(path)
        matches = re.findall(
            r"(?:^|[_-])(?:w|window[_-]?|window_size[_-]?)(\d+)(?=[_.-]|$)",
            path.stem, flags=re.I,
        )
        filename_sizes.update(int(v) for v in matches)
        # Only read the window-size column (do not load full logs here).
        if not path.is_file():
            raise FileNotFoundError(f"Missing input file: {path}. Set the exact CSV paths in the configuration cell.")
        header = pd.read_csv(path, nrows=0).columns
        if "event_count" not in header:
            raise ValueError(f"{path.name} has no event_count column; set it from the partitioning output before training.")
        counts = pd.read_csv(path, usecols=["event_count"])["event_count"]
        if counts.empty or counts.isna().any():
            raise ValueError(f"{path.name}: event_count is missing or contains nulls.")
        counts_numeric = pd.to_numeric(counts, errors="coerce")
        if counts_numeric.isna().any() or (counts_numeric <= 0).any() or (counts_numeric % 1 != 0).any():
            raise ValueError(f"{path.name}: event_count must contain positive integers.")
        unique_counts = set(counts_numeric.astype(int).unique())
        if len(unique_counts) != 1:
            raise ValueError(f"{path.name}: variable event_count values {sorted(unique_counts)}; cannot infer a fixed window size.")
        data_sizes.update(unique_counts)
        print(f"{path.name}: {len(counts)} segments, event_count={next(iter(unique_counts))}")
    if len(data_sizes) != 1:
        raise ValueError(f"Train/validation/test disagree on event_count: {sorted(data_sizes)}")
    inferred = next(iter(data_sizes))
    if filename_sizes and filename_sizes != {inferred}:
        raise ValueError(f"Window-size marker in filenames {sorted(filename_sizes)} conflicts with event_count={inferred}")
    if explicit is not None and int(explicit) != inferred:
        raise ValueError(f"WINDOW_SIZE={explicit} conflicts with event_count={inferred} in the input CSVs")
    print(f"Resolved WINDOW_SIZE={inferred} from CSV event_count")
    return int(inferred)

WINDOW_SIZE = resolve_window_size([TRAIN_FILE, VALIDATION_FILE, TEST_FILE], WINDOW_SIZE)
DATASET_TAG = re.sub(r"[^a-z0-9-]+", "-", SYSTEM_KEY.lower()).strip("-")
EXPERIMENT_TAG = f"{DATASET_TAG}_w{WINDOW_SIZE}_{SUMMARY_TYPE}"
OUTPUT_PREFIX = f"{EXPERIMENT_TAG}_{APPROACH}"
RUN_NAME = f"{OUTPUT_PREFIX}_qwen3_8b_v3_7_4_bgl_oversampling"
MODE = "train"  # "train" or "inference_only"
# For inference_only: set the saved run directory; skip trainer/train cells.
EXISTING_RUN_DIR = OUTPUT_ROOT / RUN_NAME
INFERENCE_RESUME = True
RUN_TEST_INFERENCE = True  # Set True only after choosing settings on validation.
# ------------------------------------------------

if SYSTEM_KEY not in {"bgl", "apache", "linux", "zookeeper"}:
    raise ValueError("Invalid SYSTEM_KEY")
if APPROACH not in {"stratified", "chrono"}:
    raise ValueError("APPROACH must be stratified or chrono")
if SUMMARY_TYPE not in {"extractive", "abstractive", "hybrid"}:
    raise ValueError("Invalid SUMMARY_TYPE")
if MODE not in {"train", "inference_only"}:
    raise ValueError("Invalid MODE")
if not re.fullmatch(r"[A-Za-z0-9_.-]+", RUN_NAME):
    raise ValueError("Use a simple RUN_NAME without path separators")
RUN_DIR = OUTPUT_ROOT / RUN_NAME if MODE == "train" else Path(EXISTING_RUN_DIR)
RUN_SUMMARY_TYPE = SUMMARY_TYPE
SELECTION_METRIC = "eval_loss" if RUN_SUMMARY_TYPE == "abstractive" else "eval_critical_f1"
SELECTION_GREATER_IS_BETTER = RUN_SUMMARY_TYPE != "abstractive"
EXCLUDED_EVENT_COLUMNS = {"lineid", "id", "eventid", "label", "content", "eventtemplate", "event_template"}
LABEL_TAG = re.compile(r"\[label=[^\]\r\n]*\]", re.I)
ID_TAG = re.compile(r"\[id=([^\]\r\n]+)\]")
PIPELINE_VERSION = "logsumm-v3-7-4-bgl-auto-oversampling"
TARGET_SCHEMA = "explanation-json-v2"


def json_safe(value):
    """Convert pandas/NumPy scalar values to JSON-compatible Python values."""
    import numpy as np
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (pd.Timestamp, pd.Timedelta)):
        return str(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, default=json_safe).encode()).hexdigest()


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def save_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False, encoding="utf-8-sig")
    temporary.replace(path)


def save_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=json_safe), encoding="utf-8")
    temporary.replace(path)


def synchronize_generation_device():
    """Synchronize CUDA so wall-clock generation latency is accurate."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def empty_student_usage():
    return {
        "student_input_tokens": None,
        "student_output_tokens": 0,
        "student_total_tokens": None,
        "student_generation_time_seconds": None,
        "student_end_to_end_time_seconds": None,
        "student_tokens_per_second": None,
        "student_peak_cuda_memory_bytes": None,
        "student_eos_reached": False,
    }


def complete_student_usage(usage):
    """Derive totals/throughput from tokenizer IDs, not decoded text."""
    inp, out = usage["student_input_tokens"], usage["student_output_tokens"]
    usage["student_total_tokens"] = inp + out if inp is not None else None
    seconds = usage["student_generation_time_seconds"]
    usage["student_tokens_per_second"] = out / seconds if seconds and seconds > 0 else None
    return usage


def inference_usage_summary(predictions, status_column):
    """Totals count attempted generations, including invalid JSON and output errors."""
    attempted = pd.to_numeric(predictions["student_end_to_end_time_seconds"], errors="coerce").notna()
    output_tokens = pd.to_numeric(predictions.loc[attempted, "student_output_tokens"], errors="coerce")
    generation_seconds = pd.to_numeric(predictions.loc[attempted, "student_generation_time_seconds"], errors="coerce")
    input_tokens = pd.to_numeric(predictions.loc[attempted, "student_input_tokens"], errors="coerce")
    end_to_end = pd.to_numeric(predictions.loc[attempted, "student_end_to_end_time_seconds"], errors="coerce")
    generation_sum = generation_seconds.sum(min_count=1)
    return {
        "attempted_segments": int(attempted.sum()),
        "completed_segments": int(predictions[status_column].eq("completed").sum()),
        "input_tokens": int(input_tokens.sum()) if input_tokens.notna().any() else None,
        "output_tokens": int(output_tokens.sum()) if output_tokens.notna().any() else None,
        "generation_time_seconds": float(generation_sum) if pd.notna(generation_sum) else None,
        "end_to_end_time_seconds": float(end_to_end.sum()) if end_to_end.notna().any() else None,
        "output_tokens_per_generation_second": (float(output_tokens.sum()) / float(generation_sum))
            if pd.notna(generation_sum) and generation_sum > 0 else None,
    }




In [ ]:
# Google Colab downloads are triggered at the END of inference only.
AUTO_DOWNLOAD_PREDICTIONS = True

def auto_download_prediction_csvs(paths):
    paths = [Path(p) for p in paths if Path(p).is_file()]
    if not AUTO_DOWNLOAD_PREDICTIONS or not paths:
        return
    try:
        from google.colab import files as colab_files
    except ImportError:
        print("Not running in Colab; generated CSVs:", [str(p) for p in paths])
        return
    for path in paths:
        print("Downloading inference CSV:", path)
        colab_files.download(str(path))


## 2. Student instructions and target validation


In [ ]:
SHARED_INSTRUCTIONS = """"""

SYSTEM_DESCRIPTIONS = {'bgl': 'IBM BlueGene/L supercomputer', 'apache': 'Apache HTTP server. For unknown events, distinguish genuine web-server or application failures from expected request handling and routine operation. Treat an event as a fault only when the logs provide clear evidence of an unsuccessful internal operation, malfunction, or abnormal service behavior. Client-side errors, rejected requests, and diagnostic messages are not faults unless they reveal an underlying system problem. Consider the event context and outcome rather than severity or error keywords alone.', 'linux': 'Linux system logs. Look for crashes and uncorrected disk/kernel faults; routine cron and successful logins are background.', 'zookeeper': 'Apache ZooKeeper service. Look for failed elections, expired sessions with impact and unrecoverable errors; routine disconnects are background.'}

PROMPTS = {
    'extractive': 'For each event, decide if it reports a real fault from its content and context. Select a fault only when it clearly shows a real failure (e.g., a crash, uncorrected hardware/I/O fault, or unsuccessful operation). Corrected or recovered notices, diagnostics, counters, periodic messages, and uncertain cases are background. Do not infer unsupported failures or outcomes.\nProvide a nonempty, brief explanation of the main selections. If no faults, begin explanation with "Everything looks normal."\ncritical_events: all distinct fault messages, one original representative each, in input order.\ncontext_events: Choose up to 3 informative background events: lifecycle transitions,\nnearby same-component context, or rare changes. Exclude repeated routine activity,\nstatus/counters, diagnostics, and already-corrected notices. Keep input order.\nCopy selected lines EXACTLY; retain [id=...].\nexplanation and lists must agree. Return JSON only:\n{"explanation":"...","critical_events":[],"context_events":[]}',
    'abstractive': 'For each event, decide if it reports a real fault from its content and context. Select a fault only when it clearly shows a real failure (e.g., a crash, uncorrected hardware/I/O fault, or unsuccessful operation). Corrected or recovered notices, diagnostics, counters, periodic messages, and uncertain cases are background. Do not infer unsupported failures or outcomes.\nWrite a brief, nonempty explanation of the main selections. If no faults, begin explanation with "Everything looks normal." Qualify inferred faults.\nWrite ONE short paragraph: \'The fault:\' followed by faults, affected components and\nobserved consequences; count repeats if useful. Then \'For background,\' with only\nqualifying context: Choose up to 3 informative background events: lifecycle transitions,\nnearby same-component context, or rare changes. Exclude repeated routine activity,\nstatus/counters, diagnostics, and already-corrected notices. Keep input order.\nIf no faults, start summary with "Everything looks normal." and optionally add relevant background.\nDo not invent outcomes or copy full lines. Return JSON only:\n{"explanation":"...","summary":"..."}',
    'hybrid': 'For each event, decide if it reports a real fault from its content and context. Select a fault only when it clearly shows a real failure (e.g., a crash, uncorrected hardware/I/O fault, or unsuccessful operation). Corrected or recovered notices, diagnostics, counters, periodic messages, and uncertain cases are background. Do not infer unsupported failures or outcomes.\nProvide a nonempty, brief explanation of the main selections. If no faults, begin explanation with "Everything looks normal." Qualify inferred faults.\ncritical_events: every distinct fault message (one exact original line per message).\ncontext_events: Choose up to 3 informative background events: lifecycle transitions,\nnearby same-component context, or rare changes. Exclude repeated routine activity,\nstatus/counters, diagnostics, and already-corrected notices. Keep input order.\nCopy selected lines exactly, preserving [id=...] and order.\nWhen critical_events is nonempty, analysis must contain:\n- fault_category: hardware, memory, storage_io, network, application, configuration, resource, security, unknown,\n- affected_components: verbatim IDs,\n- recovery_state: recovered, unrecovered, unknown,\n- likely_cause: \'undetermined\' if unsupported,\n- evidence_ids: critical-event IDs only.\nAdd a 1-2 sentence interpretation without invented consequences. If no faults, analysis=null and interpretation="Everything looks normal."; context_events can be nonempty.\nReturn JSON only:\n{"explanation":"...","critical_events":[],"context_events":[],\n"analysis":{"fault_category":"...","affected_components":[],"recovery_state":"...",\n"likely_cause":"...","evidence_ids":[]},"interpretation":"..."}',
}

# Resolve teacher-style system placeholder without changing the template wording.
PROMPTS = {name: template.replace("{SYSTEM_FULL}", SYSTEM_DESCRIPTIONS[SYSTEM_KEY])
           for name, template in PROMPTS.items()}

RUN_INSTRUCTION = (SYSTEM_DESCRIPTIONS[SYSTEM_KEY] + "\n" +
                   SHARED_INSTRUCTIONS.strip() + "\n\n" +
                   PROMPTS[RUN_SUMMARY_TYPE].strip())

def build_prompt(chunk):
    if not isinstance(chunk, str) or not chunk.strip():
        raise ValueError("Empty log chunk")
    if LABEL_TAG.search(chunk) or re.search(r"\bevent[_\s-]*template\s*=", chunk, flags=re.I):
        raise ValueError("Teacher-only field found in model input")
    return RUN_INSTRUCTION + "\n\nLog chunk:\n" + chunk + "\n\n### Response:\n"


def whitespace_key(text):
    return re.sub(r"\s+", " ", text).strip()


def source_lines(chunk):
    lines = [x for x in chunk.splitlines() if x.strip()]
    ids = [ID_TAG.findall(x) for x in lines]
    if any(len(x) != 1 for x in ids) or len({x[0] for x in ids}) != len(ids):
        raise ValueError("Each source event requires a unique [id=...] tag")
    return lines


def event_identifier(event, index):
    for key in ("LineId", "line_id", "lineid", "id"):
        if event.get(key) is not None and str(event[key]).strip():
            return str(event[key]).strip()
    return None


def validate_target(value, chunk):
    if not isinstance(value, str):
        raise ValueError("Summary must be JSON text")
    obj = json.loads(value.strip())
    required = ({"explanation", "summary"} if RUN_SUMMARY_TYPE == "abstractive" else
                {"explanation", "critical_events", "context_events"})
    if RUN_SUMMARY_TYPE == "hybrid":
        required |= {"analysis", "interpretation"}
    if not isinstance(obj, dict) or set(obj) != required:
        raise ValueError(f"Expected {sorted(required)}; received {list(obj) if isinstance(obj, dict) else type(obj)}")
    if LABEL_TAG.search(value) or re.search(r"\bevent[_\s-]*template\s*=", value, re.I):
        raise ValueError("Forbidden ground-truth/template field in summary")
    explanation = obj["explanation"]
    if not isinstance(explanation, str) or not explanation.strip():
        raise ValueError("explanation must be nonempty")
    lines = source_lines(chunk)
    if RUN_SUMMARY_TYPE == "abstractive":
        summary = obj["summary"]
        if not isinstance(summary, str) or not summary.strip():
            raise ValueError("abstractive summary must be nonempty")
        if not summary.startswith(("The fault:", "Everything looks normal.")):
            raise ValueError("Unexpected abstractive prefix")
        if summary.startswith("Everything looks normal.") and "Everything looks normal." not in explanation:
            raise ValueError("No-fault explanation requires standard phrase")
        return json.dumps(obj, ensure_ascii=False, allow_nan=False)
    lookup = {whitespace_key(line): (i, line) for i, line in enumerate(lines)}
    used = set()
    for key in ("critical_events", "context_events"):
        events = obj[key]
        if not isinstance(events, list) or any(not isinstance(x, str) or not x.strip() for x in events):
            raise ValueError(f"Invalid {key}")
        if key == "context_events" and len(events) > 3:
            raise ValueError("At most three context events")
        previous = -1
        for j, event in enumerate(events):
            match = lookup.get(whitespace_key(event))
            if match is None: raise ValueError(f"{key} contains invented or edited event")
            position, canonical = match
            if position in used or position <= previous:
                raise ValueError("Duplicate event or incorrect order")
            previous = position
            used.add(position)
            events[j] = canonical
    if not obj["critical_events"] and "Everything looks normal." not in explanation:
        raise ValueError("No-fault explanation requires standard phrase")
    if RUN_SUMMARY_TYPE == "hybrid":
        analysis = obj["analysis"]
        interp = obj["interpretation"]
        if not isinstance(interp, str) or not interp.strip():
            raise ValueError("interpretation must be nonempty")
        if not obj["critical_events"]:
            if analysis is not None or "Everything looks normal." not in interp:
                raise ValueError("Invalid no-fault hybrid output")
        else:
            fields = {"fault_category", "affected_components", "recovery_state", "likely_cause", "evidence_ids"}
            if not isinstance(analysis, dict) or set(analysis) != fields:
                raise ValueError("Incorrect analysis schema")
            if analysis["fault_category"] not in {"hardware", "memory", "storage_io", "network", "application", "configuration", "resource", "security", "unknown"}:
                raise ValueError("Invalid fault category")
            if analysis["recovery_state"] not in {"recovered", "unrecovered", "unknown"}:
                raise ValueError("Invalid recovery state")
            for field in ("affected_components", "evidence_ids"):
                if not isinstance(analysis[field], list) or any(not isinstance(t, str) for t in analysis[field]):
                    raise ValueError(f"Invalid {field}")
            ids = {ID_TAG.findall(line)[0] for line in obj["critical_events"]}
            if not set(analysis["evidence_ids"]) <= ids:
                raise ValueError("Unsupported evidence ID")
            if not isinstance(analysis["likely_cause"], str):
                raise ValueError("Invalid likely_cause")
    return json.dumps(obj, ensure_ascii=False, allow_nan=False)


def summary_payload(value):
    obj = json.loads(value) if isinstance(value, str) else value
    return obj["summary"] if RUN_SUMMARY_TYPE == "abstractive" else {k: v for k, v in obj.items() if k != "explanation"}


def has_fault_output(value):
    obj = json.loads(value) if isinstance(value, str) else value
    return obj["summary"].startswith("The fault:") if RUN_SUMMARY_TYPE == "abstractive" else bool(obj["critical_events"])


def validate_reference(value, chunk, row):
    """Validate teacher target shape and preservation. Known event labels override inference.
    Unknown-labeled events remain teacher-inferred, not ground-truth anomalies.
    Never use source annotations as student input.
    """
    canonical = validate_target(value, chunk)
    if not row.get("events_json", ""):
        return canonical
    events = json.loads(row["events_json"])
    if not isinstance(events, list): raise ValueError("Invalid events_json")
    known_fault_ids, known_normal_ids = set(), set()
    for i, event in enumerate(events):
        eid = event_identifier(event, i)
        lab = str(event.get("Label", event.get("label", "unknown"))).strip().lower()
        if eid:
            if lab == "-": known_normal_ids.add(eid)
            elif lab not in ("unknown", "", "none", "nan"): known_fault_ids.add(eid)
    obj = json.loads(canonical)
    if RUN_SUMMARY_TYPE != "abstractive":
        critical_ids = {ID_TAG.findall(line)[0] for line in obj["critical_events"]}
        context_ids = {ID_TAG.findall(line)[0] for line in obj["context_events"]}
        if known_fault_ids - critical_ids:
            # Deduplication can keep one representative of an identical message.
            by_id = {event_identifier(e,i):str(e.get("Content", "")) for i,e in enumerate(events)}
            selected_messages = {by_id.get(eid) for eid in critical_ids}
            if any(by_id.get(eid) not in selected_messages for eid in known_fault_ids - critical_ids):
                raise ValueError("Confirmed fault message omitted from critical events")
        if critical_ids & known_normal_ids or context_ids & known_fault_ids:
            raise ValueError("Teacher critical/context conflicts with known labels")
    elif known_fault_ids and not has_fault_output(obj):
        raise ValueError("Confirmed fault omitted from abstractive summary")
    return canonical


## 3. Load teacher data, oversample if needed, and retain full validation



In [ ]:

NON_BGL_BALANCE_TRAINING = True
NON_BGL_TRIGGER_FAULT_RATIO = 0.30    # 20%; edit between 0 and 1
NON_BGL_TARGET_FAULT_RATIO = 0.50     # 30%; edit between 0 and 1

if SYSTEM_KEY != "bgl":
    if not 0 <= NON_BGL_TRIGGER_FAULT_RATIO < 1:
        raise ValueError("NON_BGL_TRIGGER_FAULT_RATIO must be in [0, 1).")
    if not 0 < NON_BGL_TARGET_FAULT_RATIO < 1:
        raise ValueError("NON_BGL_TARGET_FAULT_RATIO must be in (0, 1).")
    if NON_BGL_TARGET_FAULT_RATIO < NON_BGL_TRIGGER_FAULT_RATIO:
        raise ValueError("Target ratio must be >= trigger ratio, to avoid surprising settings.")
    print(f"Non-BGL teacher-guided balancing: enabled={NON_BGL_BALANCE_TRAINING}, "
          f"trigger < {NON_BGL_TRIGGER_FAULT_RATIO:.1%}, target={NON_BGL_TARGET_FAULT_RATIO:.1%}")


In [ ]:
def get_student_input(row):
    text = row.get(INPUT_COLUMN, "")
    if isinstance(text, str) and text.strip():
        if LABEL_TAG.search(text):
            raise ValueError(f"{INPUT_COLUMN} contains event ground truth")
        if re.search(r"\bevent[_\s-]*template\s*=", text, flags=re.I):
            raise ValueError(f"{INPUT_COLUMN} contains EventTemplate metadata; regenerate teacher data with the corrected label notebook")
        source_lines(text)
        return text
    if row.get("events_json", ""):
        events = json.loads(row["events_json"])
        if not isinstance(events, list) or not events: raise ValueError("Invalid events_json")
        lines = []
        for index, event in enumerate(events):
            if not isinstance(event, dict) or not isinstance(event.get("Content"), str):
                raise ValueError("Each source event requires Content")
            eid = event_identifier(event, index)
            if not eid:
                raise ValueError("events_json lacks stable IDs; supply Content_without_labels")
            metadata = [f"{k}={v}" for k, v in event.items()
                        if k.lower().replace("_", "") not in {"lineid","id","eventid","label","content","eventtemplate"}
                        and v is not None and str(v).strip()]
            lines.append(f"[id={eid}] " + (" ".join(metadata) + " " if metadata else "") + event["Content"])
        text = "\n".join(lines)
    else:
        raise ValueError(f"Need {INPUT_COLUMN} or events_json")
    if LABEL_TAG.search(text) or re.search(r"\bevent[_\s-]*template\s*=", text, re.I):
        raise ValueError("Ground-truth or EventTemplate metadata remains in student input")
    source_lines(text)
    return text


def load_frame(path, split, require_targets=True):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Missing {split} CSV: {path}. Set the exact finalized teacher CSV path in Cell 4.")
    frame = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    if frame.empty or "segment_id" not in frame:
        raise ValueError(f"{path}: empty data or missing segment_id")
    if frame.segment_id.duplicated().any() or frame.segment_id.str.strip().eq("").any():
        raise ValueError(f"{path}: nonunique or empty segment_id")
    if "split" in frame and not frame["split"].eq(split).all():
        raise ValueError(f"{path}: incorrect split field")
    if split == "validation":
        print(f"Using all {len(frame)} validation rows (no checkpoint flag filtering)")
    if require_targets and TARGET_COLUMN not in frame:
        raise ValueError(f"{path}: missing {TARGET_COLUMN}")
    for i, row in frame.iterrows():
        try:
            chunk = get_student_input(row)
            frame.at[i, "student_input"] = chunk
            if require_targets:
                if row.get("generation_status", "completed") != "completed":
                    raise ValueError("Training/validation need generation_status='completed'; fix failed teacher rows first")
                if row.get("summary_type", RUN_SUMMARY_TYPE) != RUN_SUMMARY_TYPE:
                    raise ValueError("Teacher summary_type differs from this run")
                frame.at[i, "target_text"] = validate_reference(row[TARGET_COLUMN], chunk, row)
        except Exception as exc:
            raise ValueError(f"{path.name}, segment {row['segment_id']}: {exc}") from exc
    return frame


def event_ids(frame):
    return {eid for text in frame.student_input for eid in ID_TAG.findall(text)}


if MODE == "train":
    train_frame = load_frame(TRAIN_FILE, "train")
    validation_frame = load_frame(VALIDATION_FILE, "validation")
    if event_ids(train_frame) & event_ids(validation_frame):
        raise ValueError("Training and validation share source events")
    # The test CSV is deliberately not loaded during training/model selection.
    print(f"Training: {len(train_frame)}; validation: {len(validation_frame)}")
    print("Only prompt and completion strings will enter the trainer dataset.")
    for name, frame in [("train", train_frame), ("validation", validation_frame)]:
        if "status_gt_label" in frame:
            print(name, "chunk distribution (reporting only):", frame.status_gt_label.value_counts().to_dict())
        if RUN_SUMMARY_TYPE == "abstractive":
            print(name, "no-fault reference summaries:", int(frame.target_text.map(lambda text: summary_payload(text).startswith("Everything looks normal.")).sum()))
else:
    validation_frame = load_frame(VALIDATION_FILE, "validation")

if MODE == "train":
    # The original, unique, teacher-labeled CSV is always the source of truth.
    # Keep ALL its columns when saving the expanded file; duplicated rows retain GT.
    original_train_columns = pd.read_csv(TRAIN_FILE, nrows=0, encoding="utf-8-sig").columns.tolist()
    for frame in (train_frame, validation_frame):
        frame["teacher_fault_status"] = frame.target_text.map(
            lambda value: "anomalous" if has_fault_output(value) else "normal")

    training_exposures = train_frame.copy()
    oversampling_label_source = None
    original_class_counts = {}
    final_class_counts = {}
    if SYSTEM_KEY == "bgl" and BALANCE_TRAINING:

        if "status_gt_label" in train_frame.columns and train_frame["status_gt_label"].astype(str).str.strip().ne("").all():
            normalized = train_frame["status_gt_label"].astype(str).str.strip().str.lower()
            label_map = {
                "normal": "normal", "anomalous": "anomalous", "anomaly": "anomalous",
                "abnormal": "anomalous", "0": "normal", "1": "anomalous",
                "false": "normal", "true": "anomalous",
            }
            unknown = sorted(set(normalized) - set(label_map))
            if unknown:
                raise ValueError(f"Unknown BGL status_gt_label values: {unknown}. Do not guess oversampling labels.")
            train_frame["oversampling_class"] = normalized.map(label_map)
            oversampling_label_source = "status_gt_label"
        else:
            train_frame["oversampling_class"] = train_frame["teacher_fault_status"]
            oversampling_label_source = "teacher_fault_status (derived from GT)"
            print("WARNING: status_gt_label unavailable/incomplete; balancing BGL using teacher-summary fault presence.")

        original_class_counts = {k: int(v) for k, v in train_frame.oversampling_class.value_counts().items()}
        normal_count = original_class_counts.get("normal", 0)
        anomalous_count = original_class_counts.get("anomalous", 0)
        if normal_count == 0 or anomalous_count == 0:
            print("BGL oversampling skipped: both normal and anomalous examples are required.")
        elif anomalous_count < normal_count:
            extra = train_frame.loc[train_frame.oversampling_class.eq("anomalous")].sample(
                n=normal_count - anomalous_count, replace=True, random_state=SEED)
            training_exposures = pd.concat([train_frame, extra], ignore_index=True)
        else:
            print("BGL oversampling skipped: anomalous class is not smaller than normal class; no rows removed.")
        training_exposures = training_exposures.sample(frac=1, random_state=SEED).reset_index(drop=True)
        final_class_counts = {k: int(v) for k, v in training_exposures.oversampling_class.value_counts().items()}
        print("BGL oversampling source:", oversampling_label_source)
        print("Before oversampling:", original_class_counts)
        print("After oversampling:", final_class_counts)
    else:
        print("Training rows unchanged (BGL oversampling disabled or dataset is not BGL).")
        original_class_counts = {k: int(v) for k, v in train_frame.teacher_fault_status.value_counts().items()}
        final_class_counts = original_class_counts.copy()

    if not set(original_train_columns).issubset(training_exposures.columns):
        raise RuntimeError("Oversampled training data lost source CSV columns")
    if len(training_exposures) < len(train_frame):
        raise RuntimeError("Oversampling must not remove source rows")
    source_pair_counts = train_frame.groupby(["segment_id", TARGET_COLUMN], dropna=False).size()
    for segment_id, target in zip(training_exposures.segment_id, training_exposures[TARGET_COLUMN]):
        if (segment_id, target) not in source_pair_counts.index:
            raise RuntimeError(f"Oversampled row changed a teacher summary for segment {segment_id}")
    print(f"Training rows used: {len(training_exposures)} (unique source rows: {len(train_frame)})")
    print(f"Validation unchanged: {len(validation_frame)}")


In [ ]:
if SYSTEM_KEY != "bgl" and MODE == "train":
    import math

    def _non_bgl_distribution(frame, name):
        counts = frame["teacher_fault_status"].value_counts()
        normal = int(counts.get("normal", 0))
        faults = int(counts.get("anomalous", 0))
        total = normal + faults
        if total != len(frame):
            raise ValueError(f"{name}: unrecognized teacher_fault_status values")
        return {
            "split": name, "total": total, "normal": normal, "fault": faults,
            "normal_pct": round(100 * normal / total, 4) if total else 0,
            "fault_pct": round(100 * faults / total, 4) if total else 0,
        }

    non_bgl_original_distribution = _non_bgl_distribution(train_frame, "train_original")
    non_bgl_validation_distribution = _non_bgl_distribution(validation_frame, "validation_unchanged")
    print("Teacher-inferred distribution (NOT independent anomaly ground truth):")
    print(pd.DataFrame([non_bgl_original_distribution, non_bgl_validation_distribution]).to_string(index=False))

    n_normal = non_bgl_original_distribution["normal"]
    n_fault = non_bgl_original_distribution["fault"]
    original_fault_ratio = n_fault / len(train_frame)
    non_bgl_added_rows = 0
    non_bgl_balancing_reason = "disabled"
    oversampling_label_source = "teacher_fault_status (derived from validated GT)"
    if NON_BGL_BALANCE_TRAINING:
        if n_fault == 0:
            non_bgl_balancing_reason = "skipped: no teacher-identified fault examples"
        elif n_normal == 0:
            non_bgl_balancing_reason = "skipped: no normal examples"
        elif original_fault_ratio >= NON_BGL_TRIGGER_FAULT_RATIO:
            non_bgl_balancing_reason = "skipped: fault share meets/exceeds trigger"
        else:
            # Solve (n_fault + extra) / (n_original + extra) >= target.
            # A small epsilon avoids an unnecessary extra row from floating-point error.
            target = NON_BGL_TARGET_FAULT_RATIO
            non_bgl_added_rows = max(0, math.ceil((target * len(train_frame) - n_fault) / (1 - target) - 1e-10))
            if non_bgl_added_rows:
                extras = train_frame.loc[train_frame.teacher_fault_status.eq("anomalous")].sample(
                    n=non_bgl_added_rows, replace=True, random_state=SEED)
                training_exposures = pd.concat([train_frame, extras], ignore_index=True).sample(
                    frac=1, random_state=SEED).reset_index(drop=True)
                non_bgl_balancing_reason = "applied: fault examples duplicated with replacement"
            else:
                non_bgl_balancing_reason = "skipped: target already reached"

    original_class_counts = {"normal": n_normal, "anomalous": n_fault}
    non_bgl_final_distribution = _non_bgl_distribution(training_exposures, "train_effective")
    final_class_counts = {"normal": non_bgl_final_distribution["normal"],
                          "anomalous": non_bgl_final_distribution["fault"]}
    if len(training_exposures) != len(train_frame) + non_bgl_added_rows:
        raise RuntimeError("Non-BGL oversampling changed row count unexpectedly")
    if len(training_exposures) < len(train_frame):
        raise RuntimeError("Original training segments must not be removed")
    if not set(original_train_columns).issubset(training_exposures.columns):
        raise RuntimeError("Source CSV columns were lost")
    print(pd.DataFrame([non_bgl_final_distribution]).to_string(index=False))
    print("Non-BGL oversampling:", non_bgl_balancing_reason, "; added rows:", non_bgl_added_rows)


## 4. Save the effective training CSV and load a fresh base model and LoRA adapter



In [ ]:
# Unsloth must be imported before transformers/peft.
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import set_seed

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before loading the model")
set_seed(SEED)
VERSIONS = {name: importlib.metadata.version(name) for name in
            ["torch", "unsloth", "transformers", "trl", "peft", "datasets", "bitsandbytes"]}
print(VERSIONS)
print("GPU:", torch.cuda.get_device_name(0))
if "completion_only_loss" not in inspect.signature(SFTConfig).parameters:
    raise RuntimeError("Installed TRL does not support completion-only training; use the install cell in a fresh runtime")

if MODE == "train":
    if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
        raise FileExistsError("RUN_DIR is not empty. Choose a new RUN_NAME for a fresh experiment, or use inference_only.")
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    TRAINING_DATA_FILE = RUN_DIR / f"{OUTPUT_PREFIX}_train_oversampled.csv" if SYSTEM_KEY == "bgl" and BALANCE_TRAINING else RUN_DIR / f"{OUTPUT_PREFIX}_train_original.csv"
    save_csv(training_exposures[original_train_columns], TRAINING_DATA_FILE)
    OVERSAMPLING_REPORT = {
        "enabled": bool(SYSTEM_KEY == "bgl" and BALANCE_TRAINING),
        "source_file": str(TRAIN_FILE),
        "generated_training_file": str(TRAINING_DATA_FILE),
        "label_source": oversampling_label_source,
        "original_rows": int(len(train_frame)),
        "training_rows": int(len(training_exposures)),
        "added_rows": int(len(training_exposures) - len(train_frame)),
        "original_class_counts": original_class_counts,
        "final_class_counts": final_class_counts,
        "seed": int(SEED),
    }
    save_json(OVERSAMPLING_REPORT, RUN_DIR / "training_oversampling_report.json")
    print("Saved exact training CSV:", TRAINING_DATA_FILE)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
        dtype=None, load_in_4bit=LOAD_IN_4BIT,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_R, lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth",
        random_state=SEED, use_rslora=False,
    )
    RUN_CONFIG = {
        "pipeline_version": PIPELINE_VERSION, "target_schema": TARGET_SCHEMA, "system_key": SYSTEM_KEY, "approach": APPROACH,
        "summary_type": RUN_SUMMARY_TYPE, "instruction": RUN_INSTRUCTION,
        "model_name": MODEL_NAME, "max_seq_length": MAX_SEQ_LENGTH,
        "load_in_4bit": LOAD_IN_4BIT, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
        "learning_rate": LEARNING_RATE, "epochs": NUM_TRAIN_EPOCHS,
        "train_batch_size": TRAIN_BATCH_SIZE, "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "seed": SEED, "completion_only_loss": True, "supervise_explanation": True,
        "input_column": INPUT_COLUMN, "target_column": TARGET_COLUMN,
        "balance_training": bool(SYSTEM_KEY == "bgl" and BALANCE_TRAINING), "oversampling_report": OVERSAMPLING_REPORT,
        "generated_train_file_hash": file_hash(TRAINING_DATA_FILE), "selection_metric": SELECTION_METRIC, "selection_greater_is_better": SELECTION_GREATER_IS_BETTER,
        "validation_generation_max_new_tokens": MAX_NEW_TOKENS,
        "training_exposure_ids": training_exposures.segment_id.tolist(),
        "train_file_hash": file_hash(TRAIN_FILE), "validation_file_hash": file_hash(VALIDATION_FILE),
        "train_event_ids": sorted(event_ids(train_frame)),
        "validation_event_ids": sorted(event_ids(validation_frame)),
        "checkpoint_flag_column": None,
        "window_size": WINDOW_SIZE, "output_prefix": OUTPUT_PREFIX,
        "checkpoint_validation_segment_ids": validation_frame.segment_id.tolist(),
        "validation_partition_file_hash": None,
        "versions": VERSIONS, "resolved_base_commit": getattr(model.config, "_commit_hash", None),
    }
    RUN_CONFIG_ID = digest(RUN_CONFIG)
    save_csv(training_exposures[["segment_id", "teacher_fault_status"]], RUN_DIR / "training_exposures.csv")
    save_json(RUN_CONFIG, RUN_DIR / "run_config.json")
else:
    RUN_CONFIG = json.loads((RUN_DIR / "run_config.json").read_text())
    if (RUN_CONFIG["summary_type"] != RUN_SUMMARY_TYPE or RUN_CONFIG["instruction"] != RUN_INSTRUCTION
        or RUN_CONFIG["approach"] != APPROACH or RUN_CONFIG.get("system_key") != SYSTEM_KEY or RUN_CONFIG["pipeline_version"] != PIPELINE_VERSION
        or RUN_CONFIG.get("target_schema") != TARGET_SCHEMA):
        raise ValueError("Saved run and current prompt/type/approach do not match")
    if RUN_CONFIG.get("checkpoint_flag_column") != CHECKPOINT_FLAG_COLUMN:
        raise ValueError("Saved run's checkpoint-selection column differs")
    if RUN_CONFIG.get("checkpoint_validation_segment_ids") != validation_frame.segment_id.tolist():
        raise ValueError("Saved run's selected validation IDs differ")
    if RUN_CONFIG.get("validation_partition_file_hash") != (file_hash(VALIDATION_PARTITION_FILE) if VALIDATION_PARTITION_FILE is not None else None):
        raise ValueError("Saved run's validation partition reference differs")
    if file_hash(VALIDATION_FILE) != RUN_CONFIG["validation_file_hash"]:
        raise ValueError("Validation file differs from the saved experiment")
    MAX_SEQ_LENGTH = RUN_CONFIG["max_seq_length"]
    LOAD_IN_4BIT = RUN_CONFIG["load_in_4bit"]
    RUN_CONFIG_ID = digest(RUN_CONFIG)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(RUN_DIR / "adapter"), max_seq_length=MAX_SEQ_LENGTH,
        dtype=None, load_in_4bit=LOAD_IN_4BIT,
    )
if tokenizer.eos_token is None:
    raise ValueError("The model tokenizer needs an EOS token")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
EOS_TOKEN = tokenizer.eos_token


In [ ]:
if SYSTEM_KEY != "bgl" and MODE == "train":
    non_bgl_oversampling_applied = bool(non_bgl_added_rows > 0)
    if non_bgl_oversampling_applied:
        corrected_training_file = RUN_DIR / f"{OUTPUT_PREFIX}_train_oversampled.csv"
        if corrected_training_file.exists():
            raise FileExistsError(f"Refusing to overwrite {corrected_training_file}")
        TRAINING_DATA_FILE.rename(corrected_training_file)
        TRAINING_DATA_FILE = corrected_training_file

    OVERSAMPLING_REPORT.update({
        "enabled": bool(NON_BGL_BALANCE_TRAINING),
        "applied": non_bgl_oversampling_applied,
        "label_source": oversampling_label_source,
        "generated_training_file": str(TRAINING_DATA_FILE),
        "original_class_counts": original_class_counts,
        "final_class_counts": final_class_counts,
        "original_distribution": non_bgl_original_distribution,
        "validation_distribution_unchanged": non_bgl_validation_distribution,
        "effective_training_distribution": non_bgl_final_distribution,
        "trigger_fault_ratio": NON_BGL_TRIGGER_FAULT_RATIO,
        "target_fault_ratio": NON_BGL_TARGET_FAULT_RATIO,
        "reason": non_bgl_balancing_reason,
        "labels_are_independently_verified": False,
    })
    save_json(OVERSAMPLING_REPORT, RUN_DIR / "training_oversampling_report.json")
    NON_BGL_DISTRIBUTION_FILE = RUN_DIR / f"{OUTPUT_PREFIX}_nonbgl_fault_distribution.csv"
    save_csv(pd.DataFrame([non_bgl_original_distribution, non_bgl_final_distribution,
                           non_bgl_validation_distribution]), NON_BGL_DISTRIBUTION_FILE)
    RUN_CONFIG.update({
        "balance_training": bool(NON_BGL_BALANCE_TRAINING),
        "non_bgl_teacher_balancing": True,
        "non_bgl_oversampling_applied": non_bgl_oversampling_applied,
        "oversampling_report": OVERSAMPLING_REPORT,
        "generated_train_file_hash": file_hash(TRAINING_DATA_FILE),
        "non_bgl_trigger_fault_ratio": NON_BGL_TRIGGER_FAULT_RATIO,
        "non_bgl_target_fault_ratio": NON_BGL_TARGET_FAULT_RATIO,
        "non_bgl_distribution_file": str(NON_BGL_DISTRIBUTION_FILE),
    })
    RUN_CONFIG_ID = digest(RUN_CONFIG)
    save_json(RUN_CONFIG, RUN_DIR / "run_config.json")
    print("Saved training file:", TRAINING_DATA_FILE)
    print("Saved fault distribution:", NON_BGL_DISTRIBUTION_FILE)
    print("Updated run configuration and oversampling report")


## 5. Generate validation summaries for checkpoint selection


In [ ]:
class GenerationValidationTrainer(SFTTrainer):
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval", **kwargs):
        metrics = super().evaluate(
            eval_dataset=eval_dataset, ignore_keys=ignore_keys,
            metric_key_prefix=metric_key_prefix, **kwargs,
        )
        # Explicit alternative datasets are loss-only diagnostics, not checkpoint selection.
        if eval_dataset is not None:
            return metrics
        model_for_generation = self.model
        was_training = model_for_generation.training
        previous_cache = model_for_generation.config.use_cache
        evaluation_tag = getattr(self, "generation_evaluation_tag", None) or f"step_{self.state.global_step:06d}"
        rows = []
        tp = fp = fn = 0
        tn_chunk = fp_chunk = fn_chunk = tp_chunk = 0
        invalid = 0
        try:
            FastLanguageModel.for_inference(model_for_generation)
            model_for_generation.eval()
            model_for_generation.config.use_cache = True
            device = next(model_for_generation.parameters()).device
            for _, row in validation_frame.iterrows():
                raw, clean, error = "", "", ""
                usage = empty_student_usage()
                row_started = time.perf_counter()
                try:
                    inputs = tokenizer(
                        build_prompt(row.student_input), return_tensors="pt", truncation=False,
                    ).to(device)
                    length = inputs["input_ids"].shape[1]
                    usage["student_input_tokens"] = int(length)
                    if length + MAX_NEW_TOKENS > MAX_SEQ_LENGTH:
                        raise ValueError("Validation input plus output budget exceeds context")
                    synchronize_generation_device()
                    if torch.cuda.is_available():
                        torch.cuda.reset_peak_memory_stats()
                    generate_started = time.perf_counter()
                    try:
                        with torch.inference_mode():
                            generated = model_for_generation.generate(
                                **inputs, max_new_tokens=MAX_NEW_TOKENS,
                                do_sample=False, use_cache=True,
                                eos_token_id=tokenizer.eos_token_id,
                                pad_token_id=tokenizer.pad_token_id,
                            )[0, length:]
                    finally:
                        synchronize_generation_device()
                        usage["student_generation_time_seconds"] = time.perf_counter() - generate_started
                        if torch.cuda.is_available():
                            usage["student_peak_cuda_memory_bytes"] = int(torch.cuda.max_memory_allocated())
                    usage["student_output_tokens"] = int(len(generated))
                    raw = tokenizer.decode(generated, skip_special_tokens=True)
                    if not len(generated) or generated[-1].item() != tokenizer.eos_token_id:
                        raise ValueError("Generation did not finish with EOS")
                    clean = validate_target(raw, row.student_input)
                except Exception as exc:
                    error = f"{type(exc).__name__}: {exc}"
                    if isinstance(exc, RuntimeError) and ("CUDA" in str(exc) or "out of memory" in str(exc)):
                        raise
                usage["student_end_to_end_time_seconds"] = time.perf_counter() - row_started
                complete_student_usage(usage)
                gold = ({"fault_present"} if has_fault_output(row.target_text) else set()) if RUN_SUMMARY_TYPE == "abstractive" else critical_message_keys(row.target_text, row)
                if error:
                    invalid += 1
                    fn += len(gold)
                else:
                    pred = ({"fault_present"} if has_fault_output(clean) else set()) if RUN_SUMMARY_TYPE == "abstractive" else critical_message_keys(clean, row)
                    tp += len(pred & gold); fp += len(pred - gold); fn += len(gold - pred)
                    tp_chunk += int(bool(gold) and bool(pred))
                    fp_chunk += int(not gold and bool(pred))
                    fn_chunk += int(bool(gold) and not pred)
                    tn_chunk += int(not gold and not pred)
                rows.append({
                    "segment_id": row.segment_id, "status_gt_label": row.get("status_gt_label", "unknown"),
                    "GT": row.target_text, "student_summary": clean,
                    "raw_response": raw,
                    "student_explanation": json.loads(clean)["explanation"] if clean else "",
                    "student_summary_only": (summary_payload(clean) if RUN_SUMMARY_TYPE == "abstractive" else json.dumps(summary_payload(clean), ensure_ascii=False)) if clean else "",
                    "generation_status": "failed" if error else "completed",
                    "generation_error": error,
                    **usage,
                })
        finally:
            # Restore Unsloth training mode after generation, without updating weights.
            FastLanguageModel.for_training(model_for_generation)
            model_for_generation.config.use_cache = previous_cache
            model_for_generation.train(was_training)
            if rows:
                save_csv(pd.DataFrame(rows), RUN_DIR / "epoch_validation" / f"{evaluation_tag}.csv")
        denominator = 2 * tp + fp + fn
        f1 = 2 * tp / denominator if denominator else 0.0
        valid_count = len(rows) - invalid
        # Efficiency is measured over the actual generation attempts, including invalid outputs.
        epoch_usage = inference_usage_summary(pd.DataFrame(rows), "generation_status")
        generated_metrics = {
            f"{metric_key_prefix}_critical_f1": f1,
            f"{metric_key_prefix}_critical_precision": tp / (tp + fp) if tp + fp else 0.0,
            f"{metric_key_prefix}_critical_recall": tp / (tp + fn) if tp + fn else 0.0,
            f"{metric_key_prefix}_output_validity": valid_count / len(rows),
            f"{metric_key_prefix}_invalid_outputs": invalid,
            f"{metric_key_prefix}_critical_tp": tp,
            f"{metric_key_prefix}_critical_fp": fp,
            f"{metric_key_prefix}_critical_fn": fn,
            f"{metric_key_prefix}_chunk_accuracy_valid": (tp_chunk + tn_chunk) / valid_count if valid_count else 0.0,
            f"{metric_key_prefix}_chunk_precision_valid": tp_chunk / (tp_chunk + fp_chunk) if tp_chunk + fp_chunk else 0.0,
            f"{metric_key_prefix}_chunk_recall_valid": tp_chunk / (tp_chunk + fn_chunk) if tp_chunk + fn_chunk else 0.0,
            f"{metric_key_prefix}_chunk_f1_valid": 2 * tp_chunk / (2 * tp_chunk + fp_chunk + fn_chunk) if 2 * tp_chunk + fp_chunk + fn_chunk else 0.0,
            f"{metric_key_prefix}_chunk_tn_valid": tn_chunk,
            f"{metric_key_prefix}_chunk_fp_valid": fp_chunk,
            f"{metric_key_prefix}_chunk_fn_valid": fn_chunk,
            f"{metric_key_prefix}_chunk_tp_valid": tp_chunk,
            f"{metric_key_prefix}_student_output_tokens": epoch_usage["output_tokens"] or 0,
            f"{metric_key_prefix}_student_generation_seconds": epoch_usage["generation_time_seconds"] or 0.0,
            f"{metric_key_prefix}_student_tokens_per_second": epoch_usage["output_tokens_per_generation_second"] or 0.0,
        }
        if RUN_SUMMARY_TYPE == "abstractive":
            # Text prefixes indicate declared fault presence, not evidence extraction.
            generated_metrics = {k: v for k, v in generated_metrics.items()
                                 if not k.startswith(f"{metric_key_prefix}_critical_")}
            generated_metrics[f"{metric_key_prefix}_failed_anomalous_summaries"] = sum(
                r["generation_status"] == "failed" and has_fault_output(r["GT"]) for r in rows
            )
            generated_metrics = {k.replace("_chunk_", "_prefix_chunk_"): v for k, v in generated_metrics.items()}
        metrics.update(generated_metrics)
        self.log(generated_metrics)
        save_json(metrics, RUN_DIR / "epoch_validation" / f"{evaluation_tag}_metrics.json")
        print(f"Validation at step {self.state.global_step}: selection={SELECTION_METRIC}, "
              f"value={metrics.get(SELECTION_METRIC)}, invalid={invalid}/{len(rows)}")
        return metrics


def critical_message_keys(summary, row):
    """Compare message text, so a different occurrence of a repeated event is accepted."""
    events = json.loads(row.events_json)
    by_id = {event_identifier(event, i): event.get("Content", "") for i, event in enumerate(events)}
    return {by_id[ID_TAG.findall(line)[0]] for line in json.loads(summary)["critical_events"]}


## 6. Build balanced prompt/completion datasets and verify loss masks


In [ ]:
if MODE == "train":
    def make_completion_dataset(frame, split):
        records, lengths = [], []
        for _, row in frame.iterrows():
            prompt = build_prompt(row.student_input)
            completion = row.target_text + EOS_TOKEN
            tokens = tokenizer(prompt + completion, truncation=False)["input_ids"]
            completion_tokens = tokenizer(completion, add_special_tokens=False)["input_ids"]
            if len(tokens) > MAX_SEQ_LENGTH:
                raise ValueError(f"{split}/{row.segment_id}: {len(tokens)} tokens > MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}; no truncation applied")
            if len(completion_tokens) > MAX_NEW_TOKENS:
                raise ValueError(f"{split}/{row.segment_id}: reference JSON including explanation needs {len(completion_tokens)} completion tokens, exceeding MAX_NEW_TOKENS={MAX_NEW_TOKENS}. Increase the output budget and check MAX_SEQ_LENGTH; no truncation is applied.")
            records.append({"prompt": prompt, "completion": completion})
            lengths.append({"split": split, "segment_id": row.segment_id,
                            "total_tokens": len(tokens), "completion_tokens": len(completion_tokens)})
        return Dataset.from_list(records), lengths

    train_dataset, train_lengths = make_completion_dataset(training_exposures, "train")
    eval_dataset, eval_lengths = make_completion_dataset(validation_frame, "validation")
    length_report = pd.DataFrame(train_lengths + eval_lengths)
    save_csv(length_report, RUN_DIR / "token_lengths.csv")
    print(length_report.groupby("split")[["total_tokens", "completion_tokens"]].agg(["min", "median", "max"]))

    for _, row in validation_frame.iterrows():
        if len(tokenizer(build_prompt(row.student_input))["input_ids"]) + MAX_NEW_TOKENS > MAX_SEQ_LENGTH:
            raise ValueError(f"Validation output budget exceeds context: {row.segment_id}")

    config_kwargs = dict(
        output_dir=str(RUN_DIR / "checkpoints"),
        max_length=MAX_SEQ_LENGTH, packing=False, padding_free=False,
        completion_only_loss=True, assistant_only_loss=False,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=NUM_TRAIN_EPOCHS, max_steps=-1,
        learning_rate=LEARNING_RATE, warmup_ratio=0.1,
        weight_decay=0.01, lr_scheduler_type="linear",
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        optim="adamw_8bit", seed=SEED, data_seed=SEED,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model=SELECTION_METRIC,
        greater_is_better=SELECTION_GREATER_IS_BETTER, save_total_limit=2,
        logging_steps=1, report_to="none",
        prediction_loss_only=True, eval_accumulation_steps=1,
        dataset_num_proc=1,
    )
    # Newer TRL may default to an alternative loss; request standard token NLL when available.
    if "loss_type" in inspect.signature(SFTConfig).parameters:
        config_kwargs["loss_type"] = "nll"
    trainer = GenerationValidationTrainer(
        model=model, processing_class=tokenizer,
        train_dataset=train_dataset, eval_dataset=eval_dataset,
        args=SFTConfig(**config_kwargs),
    )

    # Verify actual trainer batches, rather than trusting configuration alone.
    for split_name, prepared, original in [
        ("train", trainer.train_dataset, train_dataset),
        ("validation", trainer.eval_dataset, eval_dataset),
    ]:
        if len(prepared) != len(original):
            raise RuntimeError("Unexpected dataset size change")
        for i, example in enumerate(prepared):
            ids = example["input_ids"]
            prompt_length = len(tokenizer(original[i]["prompt"], truncation=False)["input_ids"])
            expected_mask = [0] * prompt_length + [1] * (len(ids) - prompt_length)
            if len(ids) > MAX_SEQ_LENGTH or prompt_length >= len(ids):
                raise RuntimeError("Invalid completion length")
            if "completion_mask" in example and list(example["completion_mask"]) != expected_mask:
                raise RuntimeError("Prepared completion mask disagrees with prompt boundary")
            if ids[-1] != tokenizer.eos_token_id:
                raise RuntimeError("EOS is missing from the completion")
            if not ("completion_mask" in example or "labels" in example):
                raise RuntimeError("Neither completion_mask nor prepared labels are available")
        batch_examples = [prepared[i] for i in range(min(2, len(prepared)))]
        batch = trainer.data_collator(batch_examples)
        for i, example in enumerate(batch_examples):
            n = len(example["input_ids"])
            prompt_length = len(tokenizer(original[i]["prompt"], truncation=False)["input_ids"])
            expected = torch.tensor(example["input_ids"])
            expected[:prompt_length] = -100
            actual = batch["labels"][i, :n].cpu()
            if not torch.equal(actual, expected) or not torch.all(batch["labels"][i, n:] == -100):
                raise RuntimeError("Loss mask check failed: prompt/padding must be ignored and completion retained")
    print("Verified: prompt and padding tokens are masked; explanation, summary, and EOS are supervised.")
    ex = trainer.train_dataset[0]
    p_len = len(tokenizer(train_dataset[0]["prompt"], truncation=False)["input_ids"])
    print("Example supervised text:\n", tokenizer.decode(ex["input_ids"][p_len:]))
else:
    print("Inference-only mode: dataset/trainer creation skipped.")


## 7. Train, evaluate each epoch, and save the best adapter


In [ ]:
if MODE == "train":
    if SUMMARY_TYPE != RUN_SUMMARY_TYPE or RUN_INSTRUCTION != SYSTEM_DESCRIPTIONS[SYSTEM_KEY] + "\n" + SHARED_INSTRUCTIONS.strip() + "\n\n" + PROMPTS[SUMMARY_TYPE].strip():
        raise ValueError("Configuration changed mid-run; restart from a fresh base model")
    if RUN_CONFIG["checkpoint_validation_segment_ids"] != validation_frame.segment_id.tolist():
        raise ValueError("Checkpoint validation selection changed since trainer creation")
    if RUN_CONFIG["validation_partition_file_hash"] != (file_hash(VALIDATION_PARTITION_FILE) if VALIDATION_PARTITION_FILE is not None else None):
        raise ValueError("Validation partition reference changed since trainer creation")
    if file_hash(TRAIN_FILE) != RUN_CONFIG["train_file_hash"] or file_hash(VALIDATION_FILE) != RUN_CONFIG["validation_file_hash"]:
        raise ValueError("Input files changed after trainer preparation")
    if globals().get("TRAINING_ALREADY_STARTED", False):
        raise RuntimeError("Training already started in this session; restart with a fresh model and new RUN_NAME.")
    TRAINING_ALREADY_STARTED = True
    model.config.use_cache = False
    # Record validation loss and type-appropriate generation diagnostics each epoch.
    trainer.generation_evaluation_tag = "initial"
    baseline_validation = trainer.evaluate()
    trainer.generation_evaluation_tag = None
    save_json(baseline_validation, RUN_DIR / "initial_validation_metrics.json")
    trainer_stats = trainer.train()
    trainer.generation_evaluation_tag = "selected_checkpoint"
    best_validation = trainer.evaluate()
    trainer.generation_evaluation_tag = None
    save_json(best_validation, RUN_DIR / "best_validation_metrics.json")
    save_csv(pd.DataFrame(trainer.state.log_history), RUN_DIR / "training_history.csv")
    adapter_dir = RUN_DIR / "adapter"
    model.save_pretrained(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))
    adapter_files = sorted(p for p in adapter_dir.iterdir() if p.is_file())
    model_fingerprint = digest({p.name: file_hash(p) for p in adapter_files})
    SAVE_METADATA = {
        "run_config_id": RUN_CONFIG_ID, "model_fingerprint": model_fingerprint,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "selection_metric": SELECTION_METRIC, "best_validation_metric": trainer.state.best_metric,
        "global_step": trainer.state.global_step,
    }
    save_json(SAVE_METADATA, RUN_DIR / "saved_model.json")
    print("Saved selected adapter:", adapter_dir)
    print("Best checkpoint:", trainer.state.best_model_checkpoint)
    print("Best trained checkpoint selected by", SELECTION_METRIC,
          "(higher is better)" if SELECTION_GREATER_IS_BETTER else "(lower is better)")
    if RUN_SUMMARY_TYPE != "abstractive" and trainer.state.best_metric == 0:
        print("WARNING: No trained checkpoint achieved positive validation critical-event F1.")
else:
    SAVE_METADATA = json.loads((RUN_DIR / "saved_model.json").read_text())
    adapter_dir = RUN_DIR / "adapter"
    observed = digest({p.name: file_hash(p) for p in sorted(adapter_dir.iterdir()) if p.is_file()})
    if observed != SAVE_METADATA["model_fingerprint"] or RUN_CONFIG_ID != SAVE_METADATA["run_config_id"]:
        raise ValueError("Saved adapter/configuration fingerprint mismatch")


## 8. Generate best-checkpoint validation summaries and measure efficiency



In [ ]:
def extract_summary_json(raw, chunk):
    """Find first complete JSON object matching the required summary schema.

    Preserve original text separately. Ignore braces within quoted JSON strings.
    Never fabricate, repair or merge generated JSON objects.
    """
    decoder = json.JSONDecoder()
    required = ({"explanation", "summary"} if RUN_SUMMARY_TYPE == 'abstractive'
                else {"explanation", "critical_events", "context_events"})
    if RUN_SUMMARY_TYPE == 'hybrid':
        required |= {"analysis", "interpretation"}
    first_schema_json = ''
    first_validation_error = ''
    for match in re.finditer(r'\{', raw):
        try:
            obj, end = decoder.raw_decode(raw, match.start())
        except (json.JSONDecodeError, ValueError):
            continue
        if not isinstance(obj, dict) or set(obj) != required:
            continue
        candidate = raw[match.start():end]
        if not first_schema_json:
            first_schema_json = candidate
        try:
            validated = validate_target(candidate, chunk)
            return candidate, validated, '', True
        except Exception as exc:
            if not first_validation_error:
                first_validation_error = f'{type(exc).__name__}: {exc}'
    if first_schema_json:
        return first_schema_json, '', first_validation_error, True
    return '', '', 'No complete schema-matching JSON object found', False


FastLanguageModel.for_inference(model)
model.config.use_cache = True
model.eval()


def inference_id():
    return digest({"model": SAVE_METADATA["model_fingerprint"], "run": RUN_CONFIG_ID,
                   "instruction": RUN_INSTRUCTION, "max_new_tokens": MAX_NEW_TOKENS,
                   "do_sample": False, "json_extraction": True, "pipeline_version": PIPELINE_VERSION + "-parity-v1"})


def generate_one(chunk):
    """Return decoded output, validation status, and per-attempt token/time usage.

    GPU timing measures model.generate (with synchronization); end-to-end includes
    prompt tokenization, decoding and output validation. Counts are token IDs,
    including EOS and any other generated special tokens.
    """
    usage = empty_student_usage()
    started = time.perf_counter()
    raw, clean, status, error = "", "", "failed", ""
    extracted_json = ""
    json_structurally_valid = False
    try:
        if SUMMARY_TYPE != RUN_SUMMARY_TYPE:
            raise ValueError("SUMMARY_TYPE changed; restart with the intended model")
        inputs = tokenizer(build_prompt(chunk), return_tensors="pt", truncation=False).to(next(model.parameters()).device)
        input_length = int(inputs["input_ids"].shape[1])
        usage["student_input_tokens"] = input_length
        if input_length + MAX_NEW_TOKENS > MAX_SEQ_LENGTH:
            raise ValueError(
                f"Input {input_length} + output budget {MAX_NEW_TOKENS} exceeds context {MAX_SEQ_LENGTH}; no truncation applied"
            )
        synchronize_generation_device()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        generation_started = time.perf_counter()
        try:
            with torch.inference_mode():
                result = model.generate(
                    **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                    use_cache=True, eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.pad_token_id,
                )
        finally:
            synchronize_generation_device()
            usage["student_generation_time_seconds"] = time.perf_counter() - generation_started
            if torch.cuda.is_available():
                usage["student_peak_cuda_memory_bytes"] = int(torch.cuda.max_memory_allocated())
        generated = result[0, input_length:]
        usage["student_output_tokens"] = int(generated.numel())
        raw = tokenizer.decode(generated, skip_special_tokens=True)
        finished = len(generated) > 0 and generated[-1].item() == tokenizer.eos_token_id
        usage["student_eos_reached"] = bool(finished)
        if not raw.strip():
            error = "Empty response; all types require a JSON object with nonempty explanation"
        else:
            extracted_json, clean, error, json_structurally_valid = extract_summary_json(raw, chunk)
            if clean:
                status = "completed"
            elif not finished:
                error += " | Generation did not reach EOS (possibly output-budget exhaustion)"
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        if isinstance(exc, RuntimeError) and ("CUDA" in str(exc) or "out of memory" in str(exc)):
            raise
    finally:
        usage["student_end_to_end_time_seconds"] = time.perf_counter() - started
        complete_student_usage(usage)
    return raw, extracted_json, clean, status, error, json_structurally_valid, usage


def process_inference(frame, split):
    output_col = f"student_{RUN_SUMMARY_TYPE}"
    raw_col = output_col + "_raw"
    json_col = output_col + "_json_extracted"
    schema_col = output_col + "_json_structurally_valid"
    status_col = output_col + "_status"
    error_col = output_col + "_error"
    output = RUN_DIR / f"{OUTPUT_PREFIX}_{split}_predictions.csv"
    checkpoint = RUN_DIR / f"{OUTPUT_PREFIX}_{split}_checkpoint.csv"
    signature = inference_id()
    base = frame.copy()
    explanation_col = output_col + "_explanation"
    payload_col = output_col + "_without_explanation"
    usage_columns = list(empty_student_usage())
    added = [output_col, raw_col, json_col, schema_col, status_col, error_col, explanation_col, payload_col, "inference_config", *usage_columns]
    def store_parts(index, clean):
        results.at[index, explanation_col] = json.loads(clean)["explanation"] if clean else ""
        payload = summary_payload(clean) if clean else ""
        results.at[index, payload_col] = (json.dumps(payload, ensure_ascii=False) if clean and RUN_SUMMARY_TYPE != "abstractive" else payload)
    base = base.drop(columns=added, errors="ignore")
    base = base.drop(columns=["target_text"], errors="ignore")
    results = base.copy()
    for column in added:
        results[column] = ""
    results[status_col] = "pending"
    results["inference_config"] = signature
    resume_path = checkpoint if checkpoint.exists() else output
    if INFERENCE_RESUME and resume_path.exists():
        old = pd.read_csv(resume_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
        if (not set(results.columns).issubset(old.columns)
            or not old[base.columns].equals(base)
            or not old["inference_config"].eq(signature).all()):
            raise ValueError("Inference checkpoint differs in input, model, prompt, or generation settings. Use a new run/path or INFERENCE_RESUME=False.")
        results = old[results.columns].copy()
    try:
        for i, row in base.iterrows():
            if results.at[i, status_col] == "completed":
                try:
                    clean = validate_target(results.at[i, output_col], row.student_input)
                    results.at[i, output_col] = clean
                    store_parts(i, clean)
                    continue
                except Exception:
                    pass
            print(f"{split}: {i + 1}/{len(base)} | {row.segment_id}", flush=True)
            try:
                raw, extracted_json, clean, status, error, json_structurally_valid, usage = generate_one(row.student_input)
            except Exception as exc:
                raw, extracted_json, clean, status, error, json_structurally_valid = "", "", "", "failed", f"{type(exc).__name__}: {exc}", False
                usage = empty_student_usage()
                # CUDA failures often poison the context; save progress and stop.
                if isinstance(exc, RuntimeError) and ("CUDA" in str(exc) or "out of memory" in str(exc)):
                    results.loc[i, [raw_col, json_col, schema_col, output_col, status_col, error_col]] = [raw, extracted_json, json_structurally_valid, clean, status, error]
                    for key, value in usage.items():
                        results.at[i, key] = value
                    raise
            results.loc[i, [raw_col, json_col, schema_col, output_col, status_col, error_col]] = [raw, extracted_json, json_structurally_valid, clean, status, error]
            for key, value in usage.items():
                results.at[i, key] = value
            store_parts(i, clean)
            save_csv(results, checkpoint)
    finally:
        save_csv(results, checkpoint)
        save_csv(results, output)
    print(results[status_col].value_counts().to_dict(), output)
    print("Student inference usage:", inference_usage_summary(results, status_col))
    return results


validation_predictions = process_inference(validation_frame, "validation")


## 9. Evaluate generated validation summaries


In [ ]:
def evaluate_predictions(predictions, split):
    """Diagnostics against teacher summaries, not a separate anomaly detector.
    Failed outputs remain in coverage denominators, rather than being dropped.
    Event agreement uses message text so choosing a different occurrence of a
    repeated message does not count as an error.
    """
    out = f"student_{RUN_SUMMARY_TYPE}"
    valid = predictions[out + "_status"].eq("completed")
    metrics = {"split": split, "chunks": len(predictions),
               "valid_outputs": int(valid.sum()), "valid_output_rate": float(valid.mean())}
    matched = 0
    full_matched = 0
    explanation_matched = 0
    tp = fp = fn = 0
    chunk_tp = chunk_fp = chunk_fn = chunk_tn = 0
    failed_with_critical = 0
    for i, row in predictions.iterrows():
        if row.get("generation_status", "completed") != "completed" or row.get("summary_type", RUN_SUMMARY_TYPE) != RUN_SUMMARY_TYPE:
            raise ValueError("Metrics require completed teacher summaries of the same type")
        reference = validate_reference(row[TARGET_COLUMN], row.student_input, row)
        predicted = row[out] if valid.loc[i] else None
        if predicted is not None:
            predicted = validate_target(predicted, row.student_input)
            matched += int(summary_payload(predicted) == summary_payload(reference))
            full_matched += int(json.loads(predicted) == json.loads(reference))
            explanation_matched += int(json.loads(predicted)["explanation"] == json.loads(reference)["explanation"])
        if RUN_SUMMARY_TYPE == "abstractive":
            actual = has_fault_output(reference)
            if predicted is None:
                failed_with_critical += int(actual)
            else:
                present = has_fault_output(predicted)
                chunk_tp += int(actual and present); chunk_fp += int(not actual and present)
                chunk_fn += int(actual and not present); chunk_tn += int(not actual and not present)
            continue
        message_by_id = {event_identifier(e, j): e.get("Content", "") for j, e in enumerate(json.loads(row["events_json"]))} if row.get("events_json", "") else {}
        def keys(summary):
            return {message_by_id.get(ID_TAG.findall(line)[0], whitespace_key(line))
                    for line in json.loads(summary)["critical_events"]}
        gold = keys(reference)
        if predicted is None:
            fn += len(gold)
            failed_with_critical += int(bool(gold))
            continue
        pred = keys(predicted)
        tp += len(gold & pred); fp += len(pred - gold); fn += len(gold - pred)
        chunk_tp += int(bool(gold) and bool(pred)); chunk_fp += int(not gold and bool(pred))
        chunk_fn += int(bool(gold) and not pred); chunk_tn += int(not gold and not pred)
    metrics["teacher_exact_match_rate"] = matched / len(predictions)  # Summary payload only.
    metrics["summary_only_exact_match_rate"] = matched / len(predictions)
    metrics["full_response_exact_match_rate"] = full_matched / len(predictions)
    metrics["explanation_exact_match_rate"] = explanation_matched / len(predictions)  # Descriptive, not explanation quality.
    if RUN_SUMMARY_TYPE != "abstractive":
        metrics.update(
            critical_message_precision=tp / (tp + fp) if tp + fp else None,
            critical_message_recall=tp / (tp + fn) if tp + fn else None,
            critical_message_f1=2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else None,
            critical_tp=tp, critical_fp=fp, critical_fn=fn,
            valid_chunk_critical_presence_tp=chunk_tp, valid_chunk_critical_presence_fp=chunk_fp,
            valid_chunk_critical_presence_fn=chunk_fn, valid_chunk_critical_presence_tn=chunk_tn,
            failed_chunks_with_teacher_critical_events=failed_with_critical,
        )
    if RUN_SUMMARY_TYPE == "abstractive":
        denom = 2 * chunk_tp + chunk_fp + chunk_fn
        metrics.update(
            valid_prefix_chunk_tp=chunk_tp, valid_prefix_chunk_fp=chunk_fp,
            valid_prefix_chunk_fn=chunk_fn, valid_prefix_chunk_tn=chunk_tn,
            valid_prefix_chunk_precision=chunk_tp / (chunk_tp + chunk_fp) if chunk_tp + chunk_fp else None,
            valid_prefix_chunk_recall=chunk_tp / (chunk_tp + chunk_fn) if chunk_tp + chunk_fn else None,
            valid_prefix_chunk_f1=2 * chunk_tp / denom if denom else None,
            valid_prefix_chunk_accuracy=(chunk_tp + chunk_tn) / int(valid.sum()) if valid.sum() else None,
            failed_anomalous_summaries=failed_with_critical,
        )
    metrics["student_inference_efficiency"] = inference_usage_summary(predictions, out + "_status")
    save_json(metrics, RUN_DIR / f"{split}_summary_metrics.json")
    print(json.dumps(metrics, indent=2))
    if RUN_SUMMARY_TYPE == "abstractive":
        print("Exact match is descriptive only; paraphrases need factual/semantic review and downstream evaluation.")
    return metrics

validation_metrics = evaluate_predictions(validation_predictions, "validation")


## 10. Final test inference and output links


In [ ]:
# Enable only after choosing model/settings using validation.
if RUN_TEST_INFERENCE:
    test_frame = load_frame(TEST_FILE, "test", require_targets=False)
    test_ids = event_ids(test_frame)
    known = set(RUN_CONFIG["train_event_ids"]) | set(RUN_CONFIG["validation_event_ids"])
    if test_ids & known:
        raise ValueError("Test source events overlap training or validation")
    test_predictions = process_inference(test_frame, "test")
    # Targets, if available, are read only for post-generation evaluation.
    if (TARGET_COLUMN in test_predictions and "generation_status" in test_predictions
        and test_predictions.generation_status.eq("completed").all()):
        test_metrics = evaluate_predictions(test_predictions, "test")
else:
    print("Test inference skipped. Set RUN_TEST_INFERENCE=True once validation decisions are final.")

print("Run files:", RUN_DIR)
print("Adapter for later reuse:", RUN_DIR / "adapter")
try:
    from IPython.display import display, FileLink
    for path in sorted(RUN_DIR.glob("*_predictions.csv")):
        display(FileLink(str(path)))
except ImportError:
    pass

# Download test predictions when available; otherwise validation predictions.
completed_prediction_files = sorted(RUN_DIR.glob(f"{OUTPUT_PREFIX}_test_predictions.csv"))
if not completed_prediction_files:
    completed_prediction_files = sorted(RUN_DIR.glob(f"{OUTPUT_PREFIX}_validation_predictions.csv"))
auto_download_prediction_csvs(completed_prediction_files)


# 11. Saving the Fine-tuned Model

In [ ]:
import getpass
import os

os.environ["HUGGING_FACE_HUB_TOKEN"] = getpass.getpass("Token: ")
assert os.environ["HUGGING_FACE_HUB_TOKEN"]

Token: ··········


In [ ]:

HUB_USERNAME = "----"  # Change to your Hugging Face username.
HUB_REPO = f"{HUB_USERNAME}/logsumm-{DATASET_TAG}-w{WINDOW_SIZE}-{SUMMARY_TYPE}-{APPROACH}-qwen3-8b-sft"
if HUB_REPO.startswith("your-username/"):
    print("Set HUB_REPO to your repository name before uploading.")
else:
    model.push_to_hub(HUB_REPO)
    tokenizer.push_to_hub(HUB_REPO)
    print("Uploaded adapter/model and tokenizer to", HUB_REPO)


## Download a ZIP containing all generated inference files


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import zipfile

ZIP_SOURCE_DIRS = [Path(RUN_DIR)]
ZIP_DESTINATION = Path(RUN_DIR).parent / f"{RUN_DIR.name}_inference_files.zip"
ZIP_DESTINATION.parent.mkdir(parents=True, exist_ok=True)
number_of_files = 0
with zipfile.ZipFile(ZIP_DESTINATION, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for source_dir in ZIP_SOURCE_DIRS:
        if not source_dir.exists():
            print("No output directory yet:", source_dir)
            continue
        for file_path in sorted(source_dir.rglob("*")):
            if file_path.is_file() and file_path.resolve() != ZIP_DESTINATION.resolve():
                archive.write(file_path, arcname=str(Path(source_dir.name) / file_path.relative_to(source_dir)))
                number_of_files += 1
print(f"ZIP created: {ZIP_DESTINATION} | Files: {number_of_files} | Bytes: {ZIP_DESTINATION.stat().st_size}")
if number_of_files:
    try:
        from google.colab import files as colab_files
        colab_files.download(str(ZIP_DESTINATION))
    except ImportError:
        print("Download manually from:", ZIP_DESTINATION)
